# Topic: Dashboard Metrics & PSI Drift

## Definition (30-second explanation)
*   *Analogy:* Driving a car without a dashboard is fine until you run out of gas or the engine overheats. An ML dashboard is your car's instrument cluster, showing you in real-time if the model is operating safely or if the data it is consuming has suddenly changed.
*   Dashboard metrics track key performance indicators (KPIs), model health, data quality, and business impact in production environments.

## Why Interviewers Ask This
*   Models do not age like wine; they age like milk. Interviewers want to know if you can proactively detect when a model is degrading in production before the business loses money, and if you can translate technical metrics (like Precision) into business terms (like operational cost).

## Core Concepts (The 3-Layer Anatomy)
*   **The Bottleneck:** Once deployed, models silently degrade because the real-world data distribution changes over time (Data Drift), rendering the original training patterns obsolete.
*   **The Mechanism:** Dashboards monitor three distinct layers:
    1.  **Model Performance:** Accuracy, Precision, Recall, AUC.
    2.  **Data Quality & Drift:** Monitored via the Population Stability Index (PSI), which compares the distribution of model scores at training time versus production time.
    3.  **Business & System Impact:** Revenue per prediction, prediction latency, and error rates.
*   **The Trade-off:** Tracking every single feature creates dashboard clutter and alert fatigue. Engineers must isolate and track only the high-signal features and core business KPIs.

## When to Use
*   Universally required for every production model. 
*   Use PSI specifically when you need to detect score distribution shifts before the actual ground-truth labels are available.

## Advantages
*   A rising PSI score is often the very first early-warning signal that a model requires retraining.
*   Translates complex ML behaviors into actionable alerts (e.g., p99 latency > 200ms triggers a system alert).

## Limitations
*   Accuracy alone is highly misleading, especially for imbalanced classes (e.g., predicting the majority class yields high accuracy but zero recall)].

## Common Comparisons
*   **Model Metric vs. Business Metric:** 
    *   *Precision* translates to the % of fraud flags that are real (Operations team cares due to investigation costs). 
    *   *Recall* translates to the % of actual fraud caught (Risk team cares due to financial loss). 
    *   *False Positive Rate* translates to the % of good customers wrongly blocked (Customer Experience team cares).

## Common Interview Traps
*   **Failing to Connect to Business Value:** Just saying "I will monitor F1 score." *Trap:* A senior data scientist says, "A 1% drop in recall in our fraud model means approximately X fraud cases go undetected per month, costing the business $Y"].

## Python Syntax
```python
import numpy as np

# Compute Population Stability Index (PSI) to detect data drift
def compute_psi(expected, actual, buckets=10):
    # 1. Bucket the training (expected) predictions into deciles
    breakpoints = np.percentile(expected, np.linspace(0, 100, buckets + 1))
    
    # 2. Calculate the percentage of records in each bucket for both distributions
    expected_pct = np.histogram(expected, bins=breakpoints)[0] / len(expected)
    actual_pct = np.histogram(actual, bins=breakpoints)[0] / len(actual)
    
    # 3. Avoid division by zero errors
    expected_pct = np.where(expected_pct == 0, 0.0001, expected_pct)
    actual_pct = np.where(actual_pct == 0, 0.0001, actual_pct)
    
    # 4. Apply the PSI formula
    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    
    return round(psi, 4)

# Interpretation: <0.10 (Stable), 0.10-0.20 (Monitor), >0.20 (Retrain)
```

## Important Formula
The Population Stability Index (PSI) is calculated across buckets as:
$$PSI = \sum \left( (\text{Actual \%} - \text{Expected \%}) \times \ln\left(\frac{\text{Actual \%}}{\text{Expected \%}}\right) \right)$$

## 45-Second Interview Answer
"To monitor production ML health, I track three layers on my dashboard: Model Performance, Data Drift, and Business Impact. I never rely on accuracy alone; I monitor Precision and Recall because they translate directly to operational costs and financial loss. To proactively catch data drift before performance drops, I track the Population Stability Index (PSI). If PSI rises above 0.20, it signals a significant distribution shift, triggering an investigation or an automated retraining pipeline, 11."